In [1]:
# Cell 1 — setup
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)
opt    = OptionsAnalyzer(client, "SPY")

In [2]:
# Cell 2 — expiries
print(opt.expiries())


['2026-04-29', '2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-15', '2026-05-22', '2026-05-29', '2026-06-05', '2026-06-18', '2026-06-30', '2026-07-17', '2026-07-31', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15']


In [3]:
# Cell 3 — front month chain
expiry = opt.nearest_expiry(0)
print(f"Front month: {expiry}")
df = opt.chain(expiry)
print(df.head(20))

Front month: 2026-04-29
        expiry  type  strike     bid     ask  last_price  volume  \
0   2026-04-29   put   500.0    0.00    0.01        0.01    10.0   
1   2026-04-29   put   505.0    0.00    0.01        0.01    10.0   
2   2026-04-29   put   515.0    0.00    0.01        0.01     0.0   
3   2026-04-29  call   525.0  184.24  186.84      184.63    18.0   
4   2026-04-29   put   525.0    0.00    0.01        0.01     0.0   
5   2026-04-29   put   530.0    0.00    0.01        0.01    10.0   
6   2026-04-29   put   535.0    0.00    0.01        0.05     0.0   
7   2026-04-29   put   540.0    0.00    0.01        0.03   100.0   
8   2026-04-29   put   545.0    0.00    0.01        0.01     0.0   
9   2026-04-29   put   550.0    0.00    0.01        0.05    20.0   
10  2026-04-29  call   555.0  154.24  156.76      152.82     0.0   
11  2026-04-29   put   555.0    0.00    0.01        0.03     0.0   
12  2026-04-29   put   560.0    0.00    0.01        0.01     2.0   
13  2026-04-29  call   5

In [4]:
# Cell 4 — summary
print(opt.summary(expiry))

{'symbol': 'SPY', 'expiry': '2026-04-29', 'days_to_expiry': 0, 'n_strikes': 146, 'total_call_oi': 159126, 'total_put_oi': 265765, 'total_call_vol': 1186635, 'total_put_vol': 1572027, 'put_call_ratio_oi': 1.67, 'put_call_ratio_vol': 1.325, 'max_pain_strike': 711.0, 'max_oi_call_strike': np.float64(718.0), 'max_oi_put_strike': np.float64(710.0), 'avg_call_iv': 0.6757, 'avg_put_iv': 0.4568, 'iv_skew': -0.219}


In [5]:
# Cell 5 — max pain (knockout price)
print(f"Max pain: ${opt.max_pain(expiry):,.2f}")

Max pain: $711.00


In [6]:
# Cell 6 — put/call ratio
print(opt.put_call_ratio())

        expiry  days_to_expiry  call_oi   put_oi  put_call_ratio
0   2026-04-29               0   159126   265765           1.670
1   2026-04-30               1   421369   819221           1.944
2   2026-05-01               2   325894   910871           2.795
3   2026-05-04               5    55858    74061           1.326
4   2026-05-05               6    14013    20909           1.492
5   2026-05-06               7     7076    11890           1.680
6   2026-05-07               8     5952     9459           1.589
7   2026-05-08               9   189410   512262           2.705
8   2026-05-15              16   695318  2690447           3.869
9   2026-05-22              23    87243   505655           5.796
10  2026-05-29              30   217049  1215410           5.600
11  2026-06-05              37    10863    21858           2.012
12  2026-06-18              50   822130  1830710           2.227
13  2026-06-30              62   157739   289932           1.838
14  2026-07-17           

In [7]:
# Cell 7 — plots
plots.options_chain(opt, expiry).show()
plots.options_oi_profile(opt, expiry).show()
plots.options_put_call(opt).show()
plots.options_surface(opt, max_expiries=6).show()

In [8]:
# front 12 expiries (~1 year for weeklies, or full year for monthlies)
plots.options_max_pain(opt, max_expiries=12).show()

# more expiries
plots.options_max_pain(opt, max_expiries=24).show()

In [9]:
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

opt = OptionsAnalyzer(client, "SPY")

# Greeks
df = opt.greeks(opt.nearest_expiry(0))
print("Greeks columns:", df.columns.tolist())
print(df[["type","strike","delta","gamma","theta","vega","rho","greek_source"]].head(10))

# GEX
strike_gex = opt.gex_by_strike()
by_exp     = opt.gex_by_expiry()
total      = opt.gex_total()

flip     = total.get("flip_strike")
flip_str = f"${flip:,.2f}" if flip is not None else "none detected"

print(f"\nRegime:          {total['regime_label']}")
print(f"Total GEX:       ${total['total_net_gex']/1e6:.1f}M")
print(f"GEX flip:        {flip_str}")
print(f"Dominant expiry: {total.get('dominant_expiry', '—')}")

# Unusual activity
unusual = opt.unusual_activity(vol_oi_threshold=3.0, min_volume=100)
print(f"\nUnusual activity — {len(unusual)} strikes flagged")
if not unusual.empty:
    print(unusual[["type","strike","expiry","volume",
                   "open_interest","vol_oi_ratio","signal"]].head(10))

# Plots
plots.options_chain(opt, opt.nearest_expiry(0)).show()
plots.options_oi_profile(opt, opt.nearest_expiry(0)).show()
plots.options_put_call(opt, by="oi").show()
plots.options_surface(opt, max_expiries=6).show()
plots.options_max_pain(opt, max_expiries=12).show()
plots.options_gex(opt, max_expiries=12).show()
plots.options_unusual(opt, vol_oi_threshold=3.0, min_volume=100).show()

Greeks columns: ['expiry', 'type', 'strike', 'bid', 'ask', 'last_price', 'volume', 'open_interest', 'implied_volatility', 'in_the_money', 'contract', 'delta', 'gamma', 'theta', 'vega', 'rho', 'theoretical_price', 'greek_source']
   type  strike   delta     gamma   theta    vega     rho   greek_source
0   put   500.0 -0.0002  0.000012 -0.0313  0.0003 -0.0000  black_scholes
1   put   505.0 -0.0002  0.000012 -0.0291  0.0003 -0.0000  black_scholes
2   put   515.0 -0.0003  0.000017 -0.0382  0.0004 -0.0000  black_scholes
3  call   525.0  0.9721  0.000549 -3.8051  0.0238  0.0138  black_scholes
4   put   525.0 -0.0003  0.000016 -0.0322  0.0004 -0.0000  black_scholes
5   put   530.0 -0.0003  0.000016 -0.0291  0.0004 -0.0000  black_scholes
6   put   535.0 -0.0003  0.000019 -0.0335  0.0004 -0.0000  black_scholes
7   put   540.0 -0.0004  0.000023 -0.0385  0.0005 -0.0000  black_scholes
8   put   545.0 -0.0003  0.000022 -0.0346  0.0005 -0.0000  black_scholes
9   put   550.0 -0.0003  0.000021 -0.0306

In [10]:
total = opt.gex_total()
print(total.keys())

dict_keys(['symbol', 'total_call_gex', 'total_put_gex', 'total_net_gex', 'regime', 'regime_label', 'flip_strike', 'dominant_expiry'])


In [11]:
by_exp = opt.gex_by_expiry()
print(by_exp.columns.tolist())
print(by_exp.head(2))

['expiry', 'days_to_expiry', 'call_gex', 'put_gex', 'net_gex', 'abs_gex']
       expiry  days_to_expiry      call_gex       put_gex       net_gex  \
0  2026-04-29               0  2.103073e+09 -5.777018e+09 -3.673946e+09   
1  2026-04-30               1  2.133551e+09 -2.062064e+09  7.148642e+07   

        abs_gex  
0  5.981721e+09  
1  1.771139e+09  
